In [0]:
%sql
WITH T_visits AS (
SELECT
    CDE_ID
    ,to_date(cast(T1.VisitDateId as string),'yyyyMMdd') as Visit_Date
    ,T1.LocationID
    ,SUM(Net_Revenue) AS Total_Revenue
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS T1
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_item T2
    ON T2.itemskey = T1.ItemSkey
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS T3
    ON T1.LocationID = T3.LocationID
WHERE 1=1
    AND to_date(cast(T1.VisitDateId as string),'yyyyMMdd') BETWEEN '2025-07-01' AND '2025-09-30'
        AND T3.LC_Location_Type_Description = 'Cineplex Theatre'
            AND Item_Description NOT LIKE '%Delivery%'
                AND Item_Description NOT LIKE '%BP TAG%'
                    AND Item_Description NOT LIKE 'VHO MISC%'                        
                        AND Item_Description NOT LIKE 'VHO XSCAPE%'
                            AND Item_Description NOT IN ('CAKE CUTTING FEE','HOST FEE','<Unknown>','')
                                AND EIS_Sub_Group_Description NOT LIKE '%Merchandise%'
                                    AND T1.CDE_ID IS NOT NULL
                                    --AND T1.CDE_ID = 8737367
                                        AND T1.CineClubFLAG = 'Yes'
GROUP BY
    CDE_ID
    ,to_date(cast(T1.VisitDateId as string),'yyyyMMdd')
    ,T1.LocationID
HAVING
    SUM(Net_Revenue) > 0
),
 
T_actual_visits AS (
SELECT
    CDE_ID
    ,COUNT(Visit_Date) AS Total_Visits
FROM T_visits
WHERE Visit_Date BETWEEN '2025-07-01' AND '2025-09-30'
GROUP BY
    CDE_ID
HAVING
    COUNT(Visit_Date) >= 1
),
 
T_active_Jul_to_Sep AS (
SELECT
    T2.cde_id
    ,COUNT(DISTINCT invoice_id) AS Invoices
FROM cpx_dataanalytics_gold.marketing.subscription_events AS T1
LEFT JOIN
    (SELECT *
    FROM cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot
    WHERE snapshot_date_id = 20251001) AS T2
    ON T1.CC_User_Profile_ID = T2.CC_User_Profile_ID
WHERE cast(reporting_datetime as date) BETWEEN '2025-07-01' AND '2025-09-30'
AND event_type IN ('subscription_created','subscription_renewed')
GROUP BY
    CDE_ID
HAVING
    COUNT(DISTINCT invoice_id) = 3
),
Member_Data AS
(
SELECT
    T_profile.CDE_ID
    ,CC_User_Profile_ID
    ,SCENE_Membership_ID
    ,Plan_ID
    ,Renewal_Status
    ,Total_Visits
    ,recognitions_left AS Unused_CineClub_Tickets
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot AS T_profile
INNER JOIN T_active_Jul_to_Sep
    ON T_profile.CDE_ID = T_active_Jul_to_Sep.CDE_ID
INNER JOIN T_actual_visits
    ON T_profile.CDE_ID = T_actual_visits.CDE_ID
WHERE to_date(cast(snapshot_date_id as string),'yyyyMMdd') = '2025-10-01'
    AND Renewal_Status IN ('Active_Non_Renewing', 'Active', 'Active_Annnual_Upgrade')
)
 
SELECT *
FROM Member_Data
EXH_Saving AS
(
  SELECT
    CDE_ID
    ,T1.LocationID
    ,SUM(CASE WHEN CineClub_Discount_Type_ID IN (5,6) THEN Quantity ELSE 0 END) AS No_of_Discounted_Tickets
    ,SUM(CASE WHEN CineClub_Discount_Type_ID = 6 THEN Quantity ELSE 0 END) AS No_of_Redeemed_Monthly_Free_Tickets
    ,SUM(CASE WHEN CineClub_Discount_Type_ID = 2 THEN Net_Revenue ELSE 0 END) AS Discounted_Concession_Rev_Sum -- Only needed for calculating savings on concession
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS T1
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS T2
    ON T1.LocationID = T2.LocationID
WHERE CineClubFLAG = 'Yes'                                        
            AND CDE_ID IS NOT NULL
                AND to_date(cast(T1.VisitDateId as string),'yyyyMMdd') BETWEEN '2025-07-01' AND '2025-09-30'
                    AND T2.LC_Location_Type_Description = 'Cineplex Theatre'
GROUP BY                                                
    CDE_ID
    ,T1.LocationID
),
 
Gaming_Saving AS
(
    SELECT
    CDE_ID
    ,SUM(CASE WHEN CineClub_Discount_Type_ID = 3 THEN Net_Revenue ELSE 0 END) AS Discounted_Concession_Rev_Sum
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS T1
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS T2
    ON T1.LocationID = T2.LocationID
WHERE CineClubFLAG = 'Yes'                                        
            AND CDE_ID IS NOT NULL
                AND to_date(cast(T1.VisitDateId as string),'yyyyMMdd') BETWEEN '2025-07-01' AND '2025-09-30'
                    AND T2.LC_Trade_Brand_Description IN ('Rec Room', 'Playdium')
GROUP BY CDE_ID
HAVING
    SUM(CASE WHEN CineClub_Discount_Type_ID = 3 THEN Net_Revenue ELSE 0 END) > 0
)
 